# FabMob — Embeddings Ressources (CSV + PDF + Lexique)

Génère `fabmob_ressources.pkl` à partir des CSV synthèses XD, du PDF bilan et du lexique.

**Ordre :** 1 → 2 → 3 → 4 → 5 → 6

## 1. Imports

In [ ]:
!pip install openai pypdf -q

import pandas as pd
import numpy as np
import pickle
import re
from pathlib import Path
from openai import OpenAI
from google.colab import userdata, files

openai_client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
EMBED_MODEL = 'text-embedding-3-small'
print('OK')

## 2. Uploadez les fichiers dans Colab

Uploadez tous les fichiers en une fois.

In [ ]:
files.upload()
# Fichiers attendus :
# - synthèse_vehicule_ideation_proto_indus_22à24_chatbot.csv
# - synthèse_vehicule_ideation_proto_indus_22à24_chatbot-2.csv  (XD1)
# - synthèse_vehicule_ideation_proto_indus_22à24_chatbot-3.csv  (XD2)
# - Tableau_recap_XD_-_Projets_DRs.csv
# - Tableau_recap_XD_-_Projets_DRs2.csv
# - BILAN_THÉMATIQUE_de_l_extrême_défi_des_prototypes_et_de_l_industrie_des_véhicules_intermédiaires.pdf
# - lexique_XD_mobilites.md
# + tout autre fichier CSV ou PDF à ajouter
print('Fichiers disponibles :', [f.name for f in Path('.').iterdir() if f.suffix in ['.csv', '.pdf', '.md']])

## 3. Traitement des CSV

In [ ]:
resultats = []

# ── CSV 1 : Synthèse globale (tous AAP) ──
try:
    df1 = pd.read_csv('synthèse_vehicule_ideation_proto_indus_22à24_chatbot.csv',
                      encoding='latin-1', sep=None, engine='python', header=1)
    df1.columns = ['AAP', 'Wiki', 'Description', 'Tiers0', 'ActifPassif', 'Cat', 'Region', 'CasUsage', 'Suite']
    df1 = df1.dropna(subset=['Description'])
    for _, row in df1.iterrows():
        texte = (
            f"Source : Synthèse projets XD | "
            f"AAP : {row.get('AAP', '')} | "
            f"Projet : {row.get('Description', '')} | "
            f"Organisation : {row.get('Tiers0', '')} | "
            f"Catégorie véhicule : {row.get('Cat', '')} | "
            f"Région : {row.get('Region', '')} | "
            f"Cas d'usage : {row.get('CasUsage', '')} | "
            f"Suite : {row.get('Suite', '')}"
        )
        resultats.append({'title': f"Projet XD : {str(row.get('Description', ''))[:60]}",
                          'categorie': 'CSV_Synthese', 'url': '', 'texte_embedding': texte[:1500]})
    print(f'CSV synthèse globale : {len(df1)} projets')
except Exception as e:
    print(f'ERREUR CSV1 : {e}')

# ── CSV 2 : XD1 ──
try:
    df2 = pd.read_csv('synthèse_vehicule_ideation_proto_indus_22à24_chatbot-2.csv',
                      encoding='latin-1', sep=None, engine='python')
    for _, row in df2.dropna(subset=[df2.columns[1]]).iterrows():
        cols = df2.columns
        texte = f"Source : Projets XD1 | Projet : {row[cols[1]]} | Organisation : {row[cols[2]]} | Statut : {row[cols[3]]} | Catégorie : {row[cols[4]]} | Région : {row[cols[5]]} | Cas d'usage : {row[cols[6]]}"
        resultats.append({'title': f"XD1 : {str(row[cols[1]])[:60]}",
                          'categorie': 'CSV_XD1', 'url': '', 'texte_embedding': texte[:1500]})
    print(f'CSV XD1 : {len(df2)} projets')
except Exception as e:
    print(f'ERREUR CSV2 : {e}')

# ── CSV 3 : XD2 ──
try:
    df3 = pd.read_csv('synthèse_vehicule_ideation_proto_indus_22à24_chatbot-3.csv',
                      encoding='latin-1', sep=None, engine='python')
    for _, row in df3.dropna(subset=[df3.columns[1]]).iterrows():
        cols = df3.columns
        texte = f"Source : Projets XD2 | Projet : {row[cols[1]]} | Organisation : {row[cols[2]]} | Statut : {row[cols[3]]} | Catégorie : {row[cols[4]]} | Région : {row[cols[5]]} | Cas d'usage : {row[cols[6]]}"
        resultats.append({'title': f"XD2 : {str(row[cols[1]])[:60]}",
                          'categorie': 'CSV_XD2', 'url': '', 'texte_embedding': texte[:1500]})
    print(f'CSV XD2 : {len(df3)} projets')
except Exception as e:
    print(f'ERREUR CSV3 : {e}')

# ── CSV 4 : Tableau recap projets/DR ──
try:
    df4 = pd.read_csv('Tableau_recap_XD_-_Projets_DRs.csv',
                      encoding='latin-1', sep=None, engine='python')
    df4 = df4.dropna(subset=['Description projet'])
    for _, row in df4.iterrows():
        texte = (
            f"Source : Tableau récap projets XD | "
            f"Période : {row.get('Période', '')} | "
            f"Région : {row.get('Région', '')} | "
            f"AAP : {row.get('AAP', '')} | "
            f"Projet : {row.get('Description projet', '')} | "
            f"Cas d'usage : {row.get(\"Cas d'usage\", '')} | "
            f"Territoire : {row.get(\"Territoire d'expérimentation\", '')} | "
            f"Véhicules testés : {row.get('Véhicules testés', '')}"
        )
        resultats.append({'title': f"Projet : {str(row.get('Description projet', ''))[:60]}",
                          'categorie': 'CSV_Recap', 'url': '', 'texte_embedding': texte[:1500]})
    print(f'CSV Recap projets : {len(df4)} lignes')
except Exception as e:
    print(f'ERREUR CSV4 : {e}')

# ── CSV 5 : Événements ──
try:
    df5 = pd.read_csv('Tableau_recap_XD_-_Projets_DRs2.csv',
                      encoding='latin-1', sep=None, engine='python')
    df5 = df5.dropna(subset=[df5.columns[0]])
    for _, row in df5.iterrows():
        nom = row[df5.columns[0]]
        lieu = row[df5.columns[1]] if len(df5.columns) > 1 else ''
        region = row[df5.columns[2]] if len(df5.columns) > 2 else ''
        texte = f"Source : Événements XD | Événement : {nom} | Lieu : {lieu} | Région : {region}"
        resultats.append({'title': f"Événement : {str(nom)[:60]}",
                          'categorie': 'CSV_Evenements', 'url': '', 'texte_embedding': texte[:1500]})
    print(f'CSV Événements : {len(df5)} lignes')
except Exception as e:
    print(f'ERREUR CSV5 : {e}')

print(f'\nTotal CSV : {len(resultats)} entrées')

## 4. Traitement du PDF (bilan XD)

In [ ]:
from pypdf import PdfReader

PDF_FILE = 'BILAN_THÉMATIQUE_de_l_extrême_défi_des_prototypes_et_de_l_industrie_des_véhicules_intermédiaires.pdf'

try:
    reader = PdfReader(PDF_FILE)
    # Extraire tout le texte et découper en chunks de ~1500 chars
    texte_total = ''
    for page in reader.pages:
        texte_total += page.extract_text() + ' '

    # Nettoyer
    texte_total = re.sub(r'\s+', ' ', texte_total).strip()

    # Découper en chunks de ~1500 chars sur des coupures de phrases
    CHUNK_SIZE = 1500
    mots = texte_total.split('. ')
    chunk, chunks = '', []
    for m in mots:
        if len(chunk) + len(m) < CHUNK_SIZE:
            chunk += m + '. '
        else:
            if chunk:
                chunks.append(chunk.strip())
            chunk = m + '. '
    if chunk:
        chunks.append(chunk.strip())

    for j, chunk in enumerate(chunks):
        texte = f"Source : Bilan thématique XD 2025 (ADEME/France2030) | Partie {j+1}/{len(chunks)} | {chunk}"
        resultats.append({
            'title': f"Bilan XD 2025 — partie {j+1}",
            'categorie': 'PDF_Bilan',
            'url': 'https://librairie.ademe.fr/',
            'texte_embedding': texte[:2000]
        })

    print(f'PDF bilan XD : {len(reader.pages)} pages → {len(chunks)} chunks')
except Exception as e:
    print(f'ERREUR PDF : {e}')

print(f'Total après PDF : {len(resultats)} entrées')

## 5. Traitement du lexique (injecté dans le system prompt + embeddings)

In [ ]:
LEXIQUE_FILE = 'lexique_XD_mobilites.md'

try:
    with open(LEXIQUE_FILE, 'r', encoding='utf-8') as f:
        lexique_texte = f.read()

    # Découper par sections (##)
    sections = re.split(r'\n## ', lexique_texte)
    for section in sections:
        if len(section.strip()) < 50:
            continue
        titre_section = section.split('\n')[0].strip()
        # Découper en chunks de ~1500 chars
        termes = re.split(r'\n### ', section)
        chunk = ''
        for terme in termes:
            if len(chunk) + len(terme) < 1500:
                chunk += terme + '\n'
            else:
                if chunk:
                    texte = f"Source : Lexique XD Mobilités | Section : {titre_section} | {chunk.strip()}"
                    resultats.append({'title': f"Lexique XD : {titre_section[:60]}",
                                      'categorie': 'Lexique', 'url': '', 'texte_embedding': texte[:2000]})
                chunk = terme + '\n'
        if chunk:
            texte = f"Source : Lexique XD Mobilités | Section : {titre_section} | {chunk.strip()}"
            resultats.append({'title': f"Lexique XD : {titre_section[:60]}",
                              'categorie': 'Lexique', 'url': '', 'texte_embedding': texte[:2000]})

    # Sauvegarder aussi le lexique complet pour le system prompt
    with open('lexique_pour_prompt.txt', 'w') as f:
        f.write(lexique_texte[:8000])  # 8000 chars max pour le prompt

    print(f'Lexique : {len(sections)} sections traitées')
    print(f'Total final : {len(resultats)} entrées')
except Exception as e:
    print(f'ERREUR Lexique : {e}')

# Résumé par catégorie
df_res = pd.DataFrame(resultats)
print('\nRépartition :')
print(df_res['categorie'].value_counts().to_string())

## 6. Embeddings et sauvegarde

In [ ]:
def embed_openai(textes, batch_size=100):
    all_embeddings = []
    for i in range(0, len(textes), batch_size):
        batch = textes[i:i+batch_size]
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embeddings.extend([r.embedding for r in response.data])
        print(f'  {min(i+batch_size, len(textes))}/{len(textes)}...', end='\r')
    return np.array(all_embeddings).astype(np.float16)

print(f'Calcul embeddings ({len(df_res)} entrées)...')
embeddings_res = embed_openai(df_res['texte_embedding'].tolist())
print(f'\nOK {embeddings_res.shape}')

# Sauvegarder le pkl
with open('fabmob_ressources.pkl', 'wb') as f:
    pickle.dump({'df': df_res, 'embeddings': embeddings_res}, f)
print('OK fabmob_ressources.pkl')

# Sauvegarder aussi le lexique pour le system prompt
print('OK lexique_pour_prompt.txt')

files.download('fabmob_ressources.pkl')
files.download('lexique_pour_prompt.txt')